# Boundary-Interaction KV v10: matched-factorial interaction pilot
The v10 factorial method and gates are frozen on 60 newly seeded stories: 36 train, 12 validation and 12 sealed test. Counterfactual pairs reverse evidence order, and reports separate individual causal passes from complete-family eligibility. Run cells in order.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import torch

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = Path('/kaggle/working').exists()
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name)
print('VRAM GiB:', round(gpu.total_memory / 2**30, 1))
print('Platform:', 'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'other')

In [ ]:
# Mount persistent storage and locate the v10 project archive.
if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    WORK = Path('/content/bikv_work')
    MIRROR = Path('/content/drive/MyDrive/bikv_runs')
    archives = sorted(Path('/content').glob('boundary_interaction_kv_v10*.zip'), key=lambda p: p.stat().st_mtime)
    if not archives:
        print('Upload boundary_interaction_kv_v10.zip')
        uploaded = files.upload()
        archives = [Path('/content') / name for name in uploaded if name.endswith('.zip')]
elif IS_KAGGLE:
    WORK = Path('/kaggle/working/bikv_work')
    MIRROR = None
    archives = sorted(Path('/kaggle/input').rglob('boundary_interaction_kv_v10*.zip'))
else:
    WORK = Path.cwd() / 'bikv_work'
    MIRROR = None
    archives = sorted(Path.cwd().glob('boundary_interaction_kv_v10*.zip'))
if not archives:
    raise FileNotFoundError('boundary_interaction_kv_v10.zip was not found')

WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK / 'source_v10'
if SOURCE.exists():
    shutil.rmtree(SOURCE)
SOURCE.mkdir(parents=True)
with zipfile.ZipFile(archives[-1]) as archive:
    archive.extractall(SOURCE)
project_candidates = list(SOURCE.rglob('pyproject.toml'))
if len(project_candidates) != 1:
    raise RuntimeError(f'Expected one project in archive, found {len(project_candidates)}')
PROJECT = project_candidates[0].parent
if not (PROJECT / 'configs' / 'qualification.json').exists():
    raise RuntimeError('The archive is not v10: configs/qualification.json is missing')
print('Archive:', archives[-1])
print('Project:', PROJECT)

In [ ]:
# Install code and compatibility dependencies. Model weights stay on fast ephemeral disk.
os.environ['HF_HOME'] = '/content/hf_cache' if IS_COLAB else '/kaggle/working/hf_cache' if IS_KAGGLE else str(WORK / 'hf_cache')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT)], check=True)
if sys.version_info >= (3, 13):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'fire==0.7.1'], check=True)
print('Installed. HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# Scientific configuration. Keep these values frozen within one run_id.
BASE_CONFIG = PROJECT / 'configs' / 'qualification.json'
runtime = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
runtime['data_path'] = str(WORK / 'data' / 'qualification.jsonl')
runtime['output_root'] = str(WORK / 'runs')
runtime['mirror_root'] = str(MIRROR) if MIRROR else None
runtime['bundle_after_each_mask'] = bool(IS_KAGGLE)
RUNTIME_CONFIG = WORK / 'runtime_v10.json'
RUNTIME_CONFIG.write_text(json.dumps(runtime, indent=2), encoding='utf-8')

# Restore the matching Kaggle bundle if one was attached as a private Dataset.
if IS_KAGGLE:
    expected = f"{runtime['run_id']}_artifacts.zip"
    resume_archives = list(Path('/kaggle/input').rglob(expected))
    if resume_archives:
        (WORK / 'runs').mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(resume_archives[0]) as archive:
            archive.extractall(WORK / 'runs')
        print('Restored:', resume_archives[0])
print(RUNTIME_CONFIG.read_text())

In [ ]:
def run_stage(stage):
    command = [sys.executable, '-u', '-m', 'bikv.cli', stage, '--config', str(RUNTIME_CONFIG)]
    print(' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

run_stage('prepare')

## GPU stage 0: strong backbone qualification
This stage is resumable. It evaluates native chat prompting, greedy decoding, six-way candidate likelihood, both relation hops, counterfactual pairs, boundary variants, support positions and three evidence ablations.

In [ ]:
run_stage('qualify')

In [ ]:
qualification_report = WORK / 'runs' / runtime['run_id'] / 'qualification' / 'report.md'
qualification_summary = WORK / 'runs' / runtime['run_id'] / 'qualification' / 'summary.json'
print(qualification_report.read_text(encoding='utf-8'))
decision = json.loads(qualification_summary.read_text(encoding='utf-8'))['decision']
print('Decision:', decision)
if decision != 'GO_LABEL':
    print('Stop before label. Inspect qualification/samples and revise the backbone or task.')

## GPU stage 1: matched 2x2 interaction labels
Run only after `GO_LABEL`. Every adjacent boundary receives four equal-budget interventions: both, left-only, right-only and neither. The target is their difference-in-differences interaction. Only neutral train/validation stories are labelled. Every unique mask is checkpointed and restored from Drive or the Kaggle artifact bundle.

In [ ]:
if decision != 'GO_LABEL':
    raise RuntimeError(f'Qualification returned {decision}; label is intentionally blocked')
run_stage('label')

In [ ]:
# Training is CPU-light. The sealed test is blocked unless the offline validation gate passes.
run_stage('train')
model_path = WORK / 'runs' / runtime['run_id'] / 'models' / 'pair_model.json'
model_summary = json.loads(model_path.read_text(encoding='utf-8'))
print(json.dumps(model_summary['validation_metrics'], indent=2))
print('Validation gate:', model_summary['validation_gate_passed'])
if not model_summary['validation_gate_passed']:
    raise RuntimeError('Validation gate failed; sealed test remains unopened')
run_stage('evaluate')
run_stage('report')
run_stage('bundle')

In [ ]:
bundle = WORK / 'runs' / f"{runtime['run_id']}_artifacts.zip"
report = WORK / 'runs' / runtime['run_id'] / 'reports' / 'pilot_report.md'
print(report.read_text(encoding='utf-8') if report.exists() else 'Scientific report is not complete yet.')
print('Artifact bundle:', bundle)
if IS_COLAB and bundle.exists():
    from google.colab import files
    files.download(str(bundle))
elif IS_KAGGLE:
    print('Use Save Version or download the ZIP from the Kaggle Output panel.')